# 06 — Collect Activations (DeiT / DINOv2)

Extract patch-token activations from a fine-tuned ViT at a given layer and store them in HDF5.  
Output is used to train a Sparse Autoencoder (SAE) for mechanistic interpretability.

**Config for DeiT-Base** (change the 4 lines in section 1 to switch to DINOv2):
- `MODEL_NAME  = 'deit_base_patch16_224'`
- `MODEL_LABEL = 'deit_base_patch16'`
- `LAYER_IDX   = 9`
- `CKPT_PATH   = ...deit_base_patch16_best.pth`

## 0. Kaggle Setup

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru h5py
!pip install -q PyDrive2 scikit-learn

## Google Drive

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## Paths

In [ ]:
from pathlib import Path

TRAINVAL_ROOT = Path("/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K")
TEST_ROOT     = Path("/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K")
PROJECT_ROOT  = "/kaggle/working/xai-vit-medical"

# ← update this for each model
CKPT_PATH = "/kaggle/input/models/youssefnouiouar1/crt-deit/pytorch/default/1/deit_base_patch16_best.pth"

## 1. Imports & Config

**To switch to DINOv2**, change only these 3 lines:
```python
MODEL_NAME  = 'vit_base_patch14_dinov2.lvd142m'
MODEL_LABEL = 'dinov2_vitb14'
CKPT_PATH   = '...dinov2_best.pth'
```

In [ ]:
import os
import sys
import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import timm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed

# ── Dataset ──────────────────────────────────────────────────────────────────
SEED        = 42
IMAGE_SIZE  = 224
BATCH_SIZE  = 64
NUM_WORKERS = 3
VAL_RATIO   = 0.5          # 50 000 images for extraction
NUM_CLASSES = 9
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'deit_base_patch16_224'   # ← change for DINOv2
MODEL_LABEL = 'deit_base_patch16'       # ← used in output filename
LAYER_IDX   = 9                         # transformer block to hook
D_MODEL     = 768

# ── Output ───────────────────────────────────────────────────────────────────
OUT_DIR = '/kaggle/working/activations'
os.makedirs(OUT_DIR, exist_ok=True)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device      : {DEVICE}')
print(f'Model       : {MODEL_NAME}')
print(f'Layer       : blocks[{LAYER_IDX}]  (d_model={D_MODEL})')
print(f'Output dir  : {OUT_DIR}')

## 2. Dataset

Val split — resize + ImageNet normalization only, no augmentations.

In [ ]:
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=VAL_RATIO,
    random_state=SEED,
)

val_dataset = CRCHistologyDataset(
    split='val',
    splits=crc_splits,
    image_size=IMAGE_SIZE,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f'Val images  : {len(val_dataset)}')
print(f'Val batches : {len(val_loader)}')

## 3. Load Model

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)

ckpt  = torch.load(CKPT_PATH, map_location='cpu')
state = ckpt.get('model_state_dict', ckpt)
model.load_state_dict(state)
model = model.to(DEVICE)
model.eval()

print(f'Checkpoint  : {CKPT_PATH}')
print(f'Hook target : model.blocks[{LAYER_IDX}]')

## 4. Extract Activations

A forward hook captures the output of `blocks[LAYER_IDX]`.  
Shape: `(batch, 197, 768)` → we keep patch tokens only → `(batch, 196, 768)`  
(index 0 is the CLS token, indices 1–196 are patch tokens)

In [ ]:
activation_buffer = []

def hook_fn(module, input, output):
    # output: (batch, seq_len, d_model)  — seq_len = 1 CLS + 196 patches
    patch_tokens = output[:, 1:, :]                     # drop CLS token
    activation_buffer.append(patch_tokens.cpu().to(torch.float16))

hook = model.blocks[LAYER_IDX].register_forward_hook(hook_fn)

all_labels = []
all_preds  = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Extracting'):
        logits = model(images.to(DEVICE))               # hook fires here
        preds  = logits.argmax(dim=1).cpu()
        all_labels.append(labels)
        all_preds.append(preds)

hook.remove()

activations_all = torch.cat(activation_buffer, dim=0)  # (N, 196, 768) float16
labels_all      = torch.cat(all_labels,        dim=0)  # (N,)
preds_all       = torch.cat(all_preds,         dim=0)  # (N,)

# Keep only correctly classified images
correct_mask = preds_all == labels_all
activations  = activations_all[correct_mask].numpy()
labels_np    = labels_all[correct_mask].numpy()

n_total   = len(labels_all)
n_correct = int(correct_mask.sum())
print(f'Total images    : {n_total}')
print(f'Correctly pred. : {n_correct}  ({n_correct / n_total * 100:.1f}%)')
print(f'Activations     : {activations.shape}  dtype={activations.dtype}')
print(f'Labels          : {labels_np.shape}')


## 5. Save to HDF5

In [ ]:
out_path = os.path.join(OUT_DIR, f'{MODEL_LABEL}_layer{LAYER_IDX}_acts.h5')

with h5py.File(out_path, 'w') as f:
    f.create_dataset('activations', data=activations, compression='gzip', compression_opts=4)
    f.create_dataset('labels',      data=labels_np)
    f.attrs['model']       = MODEL_LABEL
    f.attrs['layer']       = LAYER_IDX
    f.attrs['d_model']     = D_MODEL
    f.attrs['num_images']  = len(labels_np)
    f.attrs['image_size']  = IMAGE_SIZE
    f.attrs['class_names'] = CLASS_NAMES

size_gb = os.path.getsize(out_path) / 1e9
print(f'Saved : {out_path}')
print(f'Size  : {size_gb:.2f} GB')

## 6. Upload to Google Drive

In [ ]:
gd_file = drive.CreateFile({'title': os.path.basename(out_path)})
gd_file.SetContentFile(out_path)
gd_file.Upload()
print(f'Uploaded : {os.path.basename(out_path)}  (id={gd_file["id"]})')